# 02 — Data Pipeline: Dataset Loading & Visualization

This notebook demonstrates the PyTorch Dataset/DataModule pipeline
and visualizes input/target pairs.

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

from weather_ml.data.dataset import WeatherBenchDataset, WeatherBenchDataModule

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)

## 1. Load DataModule

In [ ]:
dm = WeatherBenchDataModule(data_dir='../data', batch_size=16, num_workers=0)
dm.setup('fit')

print(f'Train samples: {len(dm.train_ds)}')
print(f'Val samples:   {len(dm.val_ds)}')
print(f'Test samples:  {len(dm.test_ds)}')
print(f'Channels:      {dm.train_ds.n_channels}')
print(f'Spatial shape: {dm.train_ds.spatial_shape}')

## 2. Visualize a single sample

In [ ]:
x, y = dm.train_ds[0]
print(f'Input shape:  {x.shape}')
print(f'Target shape: {y.shape}')

variables = dm.train_ds.variables
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for i, var in enumerate(variables):
    axes[0, i].imshow(x[i].numpy(), cmap='viridis', aspect='auto')
    axes[0, i].set_title(f'Input T: {var} (normalized)')
    axes[0, i].set_xlabel('Longitude')
    axes[0, i].set_ylabel('Latitude')
    
    axes[1, i].imshow(y[i].numpy(), cmap='viridis', aspect='auto')
    axes[1, i].set_title(f'Target T+6h: {var} (normalized)')
    axes[1, i].set_xlabel('Longitude')
    axes[1, i].set_ylabel('Latitude')

plt.tight_layout()
plt.show()

## 3. Visualize the difference (what the model needs to predict)

In [ ]:
diff = y - x
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for i, var in enumerate(variables):
    im = axes[i].imshow(diff[i].numpy(), cmap='RdBu_r', aspect='auto')
    axes[i].set_title(f'Difference (T+6h - T): {var}')
    plt.colorbar(im, ax=axes[i])

plt.tight_layout()
plt.show()

## 4. Latitude weights

In [ ]:
plt.figure(figsize=(8, 3))
lats = np.linspace(90, -90, 32)
plt.plot(lats, dm.train_ds.lat_weights)
plt.xlabel('Latitude')
plt.ylabel('Weight')
plt.title('Latitude weights (cosine)')
plt.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
plt.show()

## 5. Batch from DataLoader

In [ ]:
batch = next(iter(dm.train_dataloader()))
x_batch, y_batch = batch
print(f'Batch input shape:  {x_batch.shape}')
print(f'Batch target shape: {y_batch.shape}')
print(f'Input stats: mean={x_batch.mean():.3f}, std={x_batch.std():.3f}')

## 6. Normalization statistics

In [ ]:
for i, var in enumerate(variables):
    print(f'{var}: mean={dm.train_ds.mean[0, i, 0, 0]:.2f}, std={dm.train_ds.std[0, i, 0, 0]:.2f}')